# When does fusion or compilation pay off on a Cloud TPU?
Compare strict eager, fused eager and `torch.compile` on the same fixed-shape LeJEPA training task. How long does each mode take to finish its first step? Does either alternative overtake strict eager within the measured training budget? Small operations make this a useful case for testing how rapidly the host can dispatch work to the TPU. [Basile et al., TorchTPU (arXiv v5)](https://arxiv.org/abs/2610.24161v5) · [TorchTPU execution model](https://torchtpu.readthedocs.io/en/0.2.0/torchtpu_execution_model/)

The encoder sees paired observations of a two-dimensional spiral. Its hidden coordinates are available for evaluation. Compare recovery with Gaussian regularization (SIGReg) and with alignment alone to check what the timed workload actually learns.

**Published results on one TPU v5e.** For 1,001 updates, median synchronized training time across three seeds was 7.46 s in strict eager, 3.92 s in Defer and Fuse, and 5.55 s with `torch.compile`. At 6,000 updates, mean held-out R² across the three seeds was 0.969 with SIGReg and 0.003 with alignment alone. Running the cells below produces a fresh report for your chosen hardware and settings. [Published measurements and plots](https://github.com/ab-naidu/torchtpu-lejepa-spiral/blob/main/results/summary/summary.md)

**Start on a CPU** to check the entire workflow. For the device comparison, select a supported TPU runtime and set `RUN_TPU = True`. In Colab, open this notebook and run all cells. In a checkout, use the same notebook alongside the project's scripts. CPU and TPU controls use the same seed and training configuration.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import subprocess
import sys

SOURCE = Path.cwd()
bundled_sources = {}
# The following three collapsible cells contain the published experiment scripts.
# The setup cell uses them when a runtime is missing a source file.

In [ ]:
#@title Bundled source: spiral.py
bundled_sources["spiral.py"] = (
    "1e9030d59cdfe5eef9b7c475046ed959a9601a94b27029e2afe2d7640e2138ee",
    r'''"""Synthetic LeJEPA experiment from Klindt, LeCun & Balestriero (arXiv:2605.26379).

This implementation follows the authors' public 2-D experiment conventions:
https://github.com/klindtlab/lejepa-identifiability/tree/main/experiments
The training interface exposes observations only. The known coordinates are
used after training to fit and independently test an affine recovery probe.
"""

import math

import torch
from torch import nn


def spiral_observation(latent):
    """Rotate each point by pi times its Euclidean radius (Appendix H.1)."""
    x, y = latent.unbind(dim=-1)
    angle = math.pi * torch.linalg.vector_norm(latent, dim=-1)
    cosine, sine = angle.cos(), angle.sin()
    return torch.stack((x * cosine - y * sine, x * sine + y * cosine), dim=-1)


class SpiralEncoder(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2, width), nn.GELU(),
            nn.Linear(width, width), nn.GELU(),
            nn.Linear(width, width), nn.GELU(),
            nn.Linear(width, 2),
        )
        # Fixed, real-valued quadrature avoids complex accelerator operators.
        frequencies = torch.linspace(0.0, 3.0, 17)
        gaussian = torch.exp(-0.5 * frequencies.square())
        # Integrate an even integrand over [-3, 3] using doubled trapezoids.
        coefficients = torch.full_like(frequencies, 2.0 * 3.0 / 16)
        coefficients[0] *= 0.5
        coefficients[-1] *= 0.5
        self.register_buffer("frequencies", frequencies)
        self.register_buffer("gaussian", gaussian)
        self.register_buffer("quadrature", coefficients * gaussian)

    def forward(self, observations):
        return self.network(observations)


def make_encoder(width=256):
    """Construct the four-layer, two-output MLP from Appendix H.2."""
    if width < 1:
        raise ValueError("width must be positive")
    return SpiralEncoder(width)


def make_batch(batch_size, slices, generator, rho=0.95):
    """Return CPU float tensors: two observed views [2,B,2], directions [2,S].

    As in the authors' data.py, each view receives independent OU noise
    applied to the same hidden center. Its correlation with the center is
    rho; the correlation between the two views is rho**2.
    """
    if batch_size < 2 or slices < 1 or not 0 < rho < 1:
        raise ValueError("expected batch_size >= 2, slices >= 1 and 0 < rho < 1")
    centers = torch.randn(batch_size, 2, generator=generator, device="cpu")
    noise = torch.randn(2, batch_size, 2, generator=generator, device="cpu")
    latent_views = rho * centers + math.sqrt(1 - rho**2) * noise
    directions = torch.randn(2, slices, generator=generator, device="cpu")
    return spiral_observation(latent_views), nn.functional.normalize(directions, dim=0)


def objective(encoder, views, projections, weight=0.001):
    """Compute the scalar LeJEPA loss. Set weight=0 for the alignment control."""
    embeddings = encoder(views)
    # For two views, mean squared deviation from their center is 1/4 of MSE.
    alignment = 0.25 * (embeddings[0] - embeddings[1]).square().mean()
    if weight == 0:
        return alignment
    flat = embeddings.flatten(0, 1)
    angles = (flat @ projections).unsqueeze(-1) * encoder.frequencies
    real_error = angles.cos().mean(dim=0) - encoder.gaussian
    imaginary_error = angles.sin().mean(dim=0)
    squared_error = real_error.square() + imaginary_error.square()
    sigreg = (squared_error * encoder.quadrature).sum(dim=-1).mean() * flat.shape[0]
    return (1 - weight) * alignment + weight * sigreg


def _fit_and_predict(fit_features, fit_targets, test_features):
    """CPU/FP64 minimum-norm affine fit, including rank-deficient embeddings."""
    feature_mean = fit_features.mean(dim=0)
    target_mean = fit_targets.mean(dim=0)
    weights = torch.linalg.pinv(fit_features - feature_mean) @ (fit_targets - target_mean)
    return (test_features - feature_mean) @ weights + target_mean


def _r2(target, prediction):
    residual = (target - prediction).square().sum(dim=0)
    total = (target - target.mean(dim=0)).square().sum(dim=0)
    return 1 - residual / total


@torch.no_grad()
def evaluate(encoder, device, seed=1729, n_fit=2048, n_test=2048, n_plot=512):
    """Return JSON-ready metrics and the first n_plot test examples for plotting.

    The two probes (observations and embeddings) share a fit set; the R²,
    covariance statistics, and scatter coordinates all use a disjoint test set.
    """
    if min(n_fit, n_test) < 3 or n_plot < 0:
        raise ValueError("expected n_fit and n_test >= 3 and n_plot >= 0")
    generator = torch.Generator(device="cpu").manual_seed(seed)
    true = torch.randn(n_fit + n_test, 2, generator=generator, device="cpu")
    observed = spiral_observation(true)
    was_training = encoder.training
    encoder.eval()
    try:
        embeddings = encoder(observed.to(device)).cpu().double()
    finally:
        encoder.train(was_training)
    true, observed = true.double(), observed.double()
    fit_target, test_target = true.split((n_fit, n_test))
    fit_obs, test_obs = observed.split((n_fit, n_test))
    fit_emb, test_emb = embeddings.split((n_fit, n_test))
    recovered = _fit_and_predict(fit_emb, fit_target, test_emb)
    baseline = _fit_and_predict(fit_obs, fit_target, test_obs)
    r2 = _r2(test_target, recovered)
    covariance = torch.cov(test_emb.T)
    metrics = {
        "r2": r2.mean().item(),
        "r2_by_coordinate": r2.tolist(),
        "raw_r2": _r2(test_target, baseline).mean().item(),
        "covariance_eigenvalues": torch.linalg.eigvalsh(covariance).tolist(),
        "total_variance": covariance.trace().item(),
    }
    scatter = {
        name: values[:n_plot].tolist()
        for name, values in (
            ("true", test_target), ("observed", test_obs),
            ("embedding", test_emb), ("recovered", recovered),
        )
    }
    return metrics, scatter
''',
)

In [ ]:
#@title Bundled source: run_experiment.py
bundled_sources["run_experiment.py"] = (
    "d09139af2237e1fbdaa66a60b21f06158d4898530897eb71d4c523f9d61b9c2e",
    r'''"""Run one Spiral LeJEPA configuration in a fresh process and save its measurements."""

import argparse
from contextlib import nullcontext
from copy import deepcopy
import hashlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import platform
import sys
from time import perf_counter
import traceback


class UnsupportedMode(RuntimeError):
    pass


def arguments():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--device", choices=["cpu", "tpu"], default="cpu")
    parser.add_argument("--mode", choices=["eager", "fused", "compile"], default="eager")
    parser.add_argument("--output", type=Path, required=True, help="New directory for this run")
    parser.add_argument("--steps", type=int, default=100)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--width", type=int, default=64)
    parser.add_argument("--slices", type=int, default=32)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--weight", type=float, default=0.001, help="SIGReg weight")
    parser.add_argument("--lr", type=float, default=0.003)
    parser.add_argument("--weight-decay", type=float, default=0.0)
    parser.add_argument("--rho", type=float, default=0.95)
    parser.add_argument("--block-size", type=int, default=10, help="Steps per timed block")
    parser.add_argument("--threads", type=int, default=1, help="CPU intra-op threads")
    parser.add_argument("--device-label", default="", help="Observed TPU chip type, e.g. v6e-1")
    parser.add_argument("--comparison-group", default="", help="Same identifier for runs on the same allocation")
    args = parser.parse_args()
    if min(args.steps, args.width, args.slices, args.block_size, args.threads) < 1 or args.batch_size < 2:
        parser.error("steps, width, slices, block size and threads must be positive; batch size must be at least 2")
    if not (0 <= args.weight <= 1 and 0 < args.rho < 1 and math.isfinite(args.lr) and args.lr > 0
            and math.isfinite(args.weight_decay) and args.weight_decay >= 0 and 0 <= args.seed < 2**63 - 1):
        parser.error("invalid regularization, optimizer, correlation or seed")
    return args


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def setup_report(args):
    source = hashlib.sha256()
    for name in ("run_experiment.py", "spiral.py"):
        source.update(name.encode())
        source.update(Path(__file__).with_name(name).read_bytes())
    return {
        "schema_version": 1, "status": "running", "device": args.device, "mode": args.mode, "seed": args.seed,
        "config": {
            "steps": args.steps, "batch_size": args.batch_size, "width": args.width, "slices": args.slices,
            "weight": args.weight, "lr": args.lr, "weight_decay": args.weight_decay, "rho": args.rho,
            "block_size": args.block_size, "threads": args.threads, "dtype": "float32",
            "optimizer": "AdamW", "betas": [0.9, 0.999], "eps": 1e-8, "schedule": "constant",
            "data_seed": args.seed + 1, "eval_seed": 1729, "probe_fit": 2048, "probe_test": 2048,
            "scatter_samples": 512, "parity_rtol": 0.02, "parity_atol": 1e-4,
        },
        "metadata": {
            "source_sha256": source.hexdigest(), "python": platform.python_version(),
            "platform": platform.platform(), "device_label": args.device_label or ("CPU" if args.device == "cpu" else "unreported"),
            "device_label_source": "user" if args.device_label else "default",
            "comparison_group": args.comparison_group,
            "torch_version": package_version("torch"), "torch_tpu_version": package_version("torch_tpu"),
            "libtpu_version": package_version("libtpu"), "topology": os.environ.get("TORCH_TPU_TOPOLOGY"),
            "cache": {
                "tier2": os.environ.get("TORCH_TPU_TIER2_COMPILATION_CACHE") if args.device == "tpu" else None,
                "tier3_configured": bool(os.environ.get("TORCH_TPU_TIER3_COMPILATION_CACHE_ROOT")) if args.device == "tpu" else False,
            },
        },
    }


def check_parity(torch, reference, target, reference_fn, target_fn, inputs, device, config):
    reference.zero_grad(set_to_none=True)
    target.zero_grad(set_to_none=True)
    expected = reference_fn(reference, *inputs, config["weight"])
    actual = target_fn(target, *(x.to(device) for x in inputs), config["weight"])
    expected.backward()
    actual.backward()
    expected, actual = expected.detach().cpu(), actual.detach().cpu()
    tolerances = {"rtol": config["parity_rtol"], "atol": config["parity_atol"]}
    loss_ok = torch.allclose(expected, actual, **tolerances)
    grad_checks = []
    for (name, cpu_param), target_param in zip(reference.named_parameters(), target.parameters()):
        if cpu_param.grad is None or target_param.grad is None:
            raise RuntimeError(f"Missing expected gradient for {name}")
        a, b = cpu_param.grad, target_param.grad.detach().cpu()
        grad_checks.append({"name": name, "max_abs_error": (a - b).abs().max().item(),
                            "passed": torch.allclose(a, b, **tolerances)})
    return {
        "passed": loss_ok and all(row["passed"] for row in grad_checks), "loss_passed": loss_ok,
        "reference_loss": expected.item(), "target_loss": actual.item(),
        "loss_abs_error": (expected - actual).abs().item(), "gradients": grad_checks, **tolerances,
    }


def run(args, report):
    if args.device == "cpu" and args.mode != "eager":
        raise UnsupportedMode(f"{args.mode} is a TPU execution mode; CPU supports the eager reference")

    import torch
    from spiral import evaluate, make_batch, make_encoder, objective

    torch.set_num_threads(args.threads)
    torch.manual_seed(args.seed)
    torch.set_default_dtype(torch.float32)
    mode_context = nullcontext()
    if args.device == "tpu":
        if report["metadata"]["torch_tpu_version"] is None:
            raise UnsupportedMode("torch_tpu is not installed in this interpreter")
        import torch_tpu  # noqa: F401 - registers the backend if autoloading is disabled
        if (not hasattr(torch, "tpu") or not torch.accelerator.is_available()
                or torch.accelerator.current_accelerator().type != "tpu"):
            raise UnsupportedMode("TorchTPU reports no available TPU")
        report["metadata"]["device_count"] = torch.accelerator.device_count()
        report["metadata"]["runtime_device_name"] = torch.tpu.get_device_properties().name
        if not args.device_label:
            report["metadata"]["device_label"] = report["metadata"]["runtime_device_name"]
            report["metadata"]["device_label_source"] = "runtime"
        mode = torch.tpu.EagerMode.DEFER_AND_FUSE if args.mode == "fused" else torch.tpu.EagerMode.DEFER_NEVER
        mode_context = torch.tpu.set_eager_mode(mode)
    else:
        report["metadata"]["device_count"] = 1
        if not args.device_label:
            report["metadata"]["device_label"] = platform.processor() or "CPU"

    def sync():
        if args.device == "tpu":
            torch.accelerator.synchronize()

    cfg = report["config"]
    reference = make_encoder(args.width)
    target = deepcopy(reference).to(args.device)
    optimizer = torch.optim.AdamW(target.parameters(), lr=args.lr, weight_decay=args.weight_decay, foreach=False)
    data = torch.Generator(device="cpu").manual_seed(cfg["data_seed"])
    timings = {
        "scope": "Synchronized training wall time: objective, backward, AdamW and Python dispatch. "
                 "Excludes batch generation, transfers, model setup, evaluation, and parity. "
                 "Compile mode wraps the forward objective with fullgraph=True; optimizer runs outside.",
        "first_use": "First training step in this process includes first-use overhead and execution. "
                     "Not a measurement of isolated compiler time.",
        "steady_definition": "Throughput after the first full post-first-step block; residual compilation is possible.",
        "warm_intervals": [], "training_seconds": 0.0, "steady_steps_per_second": None,
    }
    report["timing"] = timings
    with mode_context:
        if args.device == "tpu":
            report["metadata"]["effective_eager_mode"] = str(torch.tpu.eager_mode)
        train_fn = objective
        if args.mode == "compile":
            import torch._dynamo
            torch._dynamo.config.suppress_errors = False
            # Create the wrapper in the first timed section so its first-call overhead is visible.
        finished = 0
        first_inputs = None
        steady_steps, steady_seconds = 0, 0.0
        while finished < args.steps:
            count = 1 if finished == 0 else min(args.block_size, args.steps - finished)
            batches = []
            for _ in range(count):
                host_batch = make_batch(args.batch_size, args.slices, data, rho=args.rho)
                if first_inputs is None:
                    first_inputs = host_batch
                batches.append(tuple(x.to(args.device) for x in host_batch))
            sync()
            started = perf_counter()
            if args.mode == "compile" and finished == 0:
                train_fn = torch.compile(objective, backend="tpu", fullgraph=True, dynamic=False)
            for views, projections in batches:
                optimizer.zero_grad(set_to_none=True)
                loss = train_fn(target, views, projections, args.weight)
                loss.backward()
                optimizer.step()
            sync()
            seconds = perf_counter() - started
            timings["training_seconds"] += seconds
            if finished == 0:
                timings["first_step_seconds"] = seconds
            else:
                timings["warm_intervals"].append({
                    "start_step": finished + 1, "end_step": finished + count, "steps": count,
                    "seconds": seconds, "steps_per_second": count / seconds,
                    "cumulative_seconds": timings["training_seconds"],
                })
                if finished > 1:
                    steady_steps += count
                    steady_seconds += seconds
            finished += count
        timings["steady_steps_per_second"] = steady_steps / steady_seconds if steady_steps else None
        timings["steady_steps"] = steady_steps
        report["final_batch_loss"] = loss.detach().cpu().item()
        report["metrics"], report["scatter"] = evaluate(
            target, args.device, seed=cfg["eval_seed"], n_fit=cfg["probe_fit"],
            n_test=cfg["probe_test"], n_plot=cfg["scatter_samples"],
        )
        # Restore identical initial parameters after measuring so the parity check cannot prewarm the run.
        target.load_state_dict(reference.state_dict())
        target.train()
        report["parity"] = check_parity(torch, reference, target, objective, train_fn, first_inputs, args.device, cfg)
        if not report["parity"]["passed"]:
            raise RuntimeError("One-step loss or gradient differs from the CPU reference beyond the recorded tolerances")


def main():
    args = arguments()
    args.output.mkdir(parents=True, exist_ok=False)
    if args.device == "tpu":
        os.environ.setdefault("TORCH_TPU_TIER2_COMPILATION_CACHE", "disabled")
    report = setup_report(args)
    try:
        run(args, report)
        report["status"] = "ok"
        json.dumps(report, allow_nan=False)
    except (Exception, KeyboardInterrupt) as exc:
        report["status"] = "unsupported" if isinstance(exc, (UnsupportedMode, NotImplementedError)) else "error"
        report["error"] = {"type": type(exc).__name__, "message": str(exc)}
        traceback.print_exc()
    # JSON null preserves the rest of a failed measurement if a diagnostic produced NaN.
    def finite(value):
        if isinstance(value, float) and not math.isfinite(value):
            return None
        if isinstance(value, dict):
            return {k: finite(v) for k, v in value.items()}
        if isinstance(value, list):
            return [finite(v) for v in value]
        return value

    path = args.output / "result.json"
    partial = path.with_suffix(".tmp")
    partial.write_text(json.dumps(finite(report), indent=2, allow_nan=False) + "\n", encoding="utf-8")
    partial.replace(path)
    print(f"{report['status']}: {path}")
    return 0 if report["status"] == "ok" else 1


if __name__ == "__main__":
    sys.exit(main())
''',
)

In [ ]:
#@title Bundled source: report_results.py
bundled_sources["report_results.py"] = (
    "c7b9958ad30dd7d31cb2f289942cf612edbfded44dc2a0b8fb36d0c8e77b7719",
    r'''"""Summarize measured LeJEPA runs and plot their held-out predictions and timings."""

import argparse
import json
import math
import os
from collections import defaultdict
from pathlib import Path
from urllib.parse import quote

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np


def scatter_figure(run, output):
    data = run["scatter"]
    true = np.asarray(data["true"])
    fig, axes = plt.subplots(1, 3, figsize=(9, 3.5), layout="constrained")
    for ax, key, title in zip(axes, ("true", "observed", "recovered"),
                              ("True coordinates", "Observed spiral", "Decoded coordinates")):
        points = np.asarray(data[key])
        dots = ax.scatter(points[:, 0], points[:, 1], c=true[:, 0], cmap="viridis",
                          s=9, alpha=0.85, linewidths=0)
        ax.set(title=title, aspect="equal", xlabel="Coordinate 1")
    extent = np.vstack((true, np.asarray(data["recovered"])))
    low, high = float(extent.min()), float(extent.max())
    pad = max(0.05, 0.05 * (high - low))
    for ax in (axes[0], axes[2]):
        ax.set(xlim=(low - pad, high + pad), ylim=(low - pad, high + pad))
    axes[0].set_ylabel("Coordinate 2")
    fig.colorbar(dots, ax=axes, shrink=0.65, label="True coordinate 1")
    fig.suptitle(f"{run['device'].upper()} {run['mode']} · weight {run['config']['weight']:g}"
                 f" · seed {run['seed']} · held-out R² {run['metrics']['r2']:.3f}")
    fig.savefig(output, dpi=150)
    plt.close(fig)


def cumulative(run):
    timing = run["timing"]
    times = {1: timing["first_step_seconds"]}
    for interval in timing["warm_intervals"]:
        times[interval["end_step"]] = interval["cumulative_seconds"]
    return times


def observed_crossover(baseline, candidate):
    """Require the same measured boundaries; report final sustained lead and its uncertainty."""
    if baseline.keys() != candidate.keys():
        return "Different timing boundaries prevent a paired comparison."
    steps = sorted(baseline)
    wins = [candidate[s] <= baseline[s] for s in steps]
    if not wins[-1]:
        return (f"Behind at step {steps[-1]}; an earlier observed lead reversed." if any(wins)
                else f"No observed break-even through step {steps[-1]}.")
    first = 1 + max((i for i, win in enumerate(wins) if not win), default=-1)
    if first == 0:
        return f"At or ahead from step {steps[0]} at all measured boundaries."
    return (f"Behind at step {steps[first-1]}, at or ahead by step {steps[first]}. "
            + (f"Still at or ahead at the sampled boundaries through step {steps[-1]}."
               if first < len(steps) - 1 else "That was the final measurement."))


def timing_figure(by_mode, output):
    fig, ax = plt.subplots(figsize=(6, 3.6), layout="constrained")
    for mode in ("eager", "fused", "compile"):
        if mode in by_mode:
            times = cumulative(by_mode[mode])
            ax.plot(list(times), list(times.values()), marker="o", markersize=4, label=mode)
    ax.set(xlabel="Completed training steps", ylabel="Cumulative training time (s)",
           title="TPU training time including the first step")
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    fig.savefig(output, dpi=150)
    plt.close(fig)


def pair_key(run):
    meta = run["metadata"]
    if not meta.get("comparison_group") or meta.get("device_label") in (None, "", "unreported"):
        return None
    fields = ("comparison_group", "device_label", "device_count", "topology",
              "runtime_device_name", "platform", "python", "torch_version", "torch_tpu_version",
              "libtpu_version", "cache", "source_sha256")
    conditions = [run["seed"], run["config"], {k: meta.get(k) for k in fields}]
    return json.dumps(conditions, sort_keys=True)


def display_number(value, precision=3):
    if isinstance(value, (int, float)) and math.isfinite(value):
        return f"{value:.{precision}f}"
    return "—"


def write_report(runs_dir, out):
    files = sorted(runs_dir.rglob("result.json"))
    if not files:
        raise ValueError(f"No result.json found under {runs_dir}")
    out.mkdir(parents=True, exist_ok=True)
    lines = ["# LeJEPA measurements", "",
             "Each row is one invocation. The first step includes initialization triggered by that step. "
             "The steady rate excludes step one and the first post-startup block. "
             "Covariance eigenvalues show the variance along both learned dimensions.", "",
             "| Run | Device / mode | Seed | SIGReg weight | Status | Raw input R² | Learned R² | "
             "Covariance eigenvalues | First step (s) | Steady steps/s |",
             "|---|---|---:|---:|---|---:|---:|---|---:|---:|"]
    successful = []
    groups = defaultdict(list)
    issues = []
    for path in files:
        run = json.loads(path.read_text(encoding="utf-8"))
        label = path.relative_to(runs_dir).parent.as_posix().replace("|", "&#124;")
        timing, metrics = run.get("timing") or {}, run.get("metrics") or {}
        status = run["status"]
        if status == "ok" and not (run.get("parity") or {}).get("passed"):
            status = "parity unverified"
        weight = run.get("config", {}).get("weight")
        eigenvalues = metrics.get("covariance_eigenvalues") or []
        spectrum = ", ".join(f"{v:.3g}" if v is not None else "—" for v in eigenvalues) or "—"
        try:
            record_link = quote(Path(os.path.relpath(path, start=out)).as_posix())
        except ValueError:  # On Windows, a relative path cannot span drives.
            record_link = path.resolve().as_uri()
        cells = [f"[{label}]({record_link})", f"{run['device']} / {run['mode']}", run.get("seed", "—"),
                 weight, status, display_number(metrics.get("raw_r2")), display_number(metrics.get("r2")),
                 spectrum,
                 display_number(timing.get("first_step_seconds")), display_number(timing.get("steady_steps_per_second"), 1)]
        lines.append("| " + " | ".join(str(v) for v in cells) + " |")
        if status != "ok":
            detail = run.get("error") or status
            issues.append(f"- **{label}** {detail}")
            continue
        successful.append((label, run))
        if run["device"] == "tpu":
            key = pair_key(run)
            if key is not None:
                groups[key].append((label, run))

    if issues:
        lines.extend(["", "## Incomplete runs", "", *issues])

    if successful:
        lines.extend(["", "## Held-out coordinates", "",
                      "Color follows the same held-out sample across panels. A linear probe fitted on a separate split "
                      "maps learned coordinates into the true coordinate system. The table also reports the raw-input "
                      "linear probe. Compare variance with recovery to diagnose collapse."])
    for i, (label, run) in enumerate(successful, start=1):
        filename = f"coordinates-{i}.png"
        scatter_figure(run, out / filename)
        lines.extend(["", f"**{label}**", "", f"![Held-out coordinates]({filename})"])

    lines.extend(["", "## Measured TPU break-even", ""])
    pairs = 0
    for members in groups.values():
        by_mode = {run["mode"]: run for _, run in members}
        if len(by_mode) != len(members):
            lines.append("Duplicate modes in one matched group; use a unique comparison group for each repeat.")
            continue
        if "eager" not in by_mode or len(by_mode) < 2:
            continue
        eager = by_mode["eager"]
        filename = f"tpu-training-{pairs + 1}.png"
        timing_figure(by_mode, out / filename)
        pairs += 1
        lines.extend([f"Matched runs: {', '.join(label for label, _ in members)}.",
                      "", f"![Cumulative TPU training time]({filename})", ""])
        for mode, candidate in by_mode.items():
            if mode != "eager":
                lines.append(f"- **{mode} vs eager** — {observed_crossover(cumulative(eager), cumulative(candidate))}")
        lines.append("")
    if not pairs:
        lines.append("No matched, parity-passing TPU eager/comparison pair is available.")
    else:
        lines.append("Each point is a synchronized timing boundary. Cumulative time includes the first call. "
                     "Input staging outside timed blocks, process setup and evaluation are excluded. "
                     "Lines connect sampled points; the exact crossover between observations is unknown.")
    report = out / "report.md"
    report.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return report


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--runs", type=Path, required=True)
    parser.add_argument("--out", type=Path, required=True)
    args = parser.parse_args()
    print(write_report(args.runs, args.out))
''',
)

In [ ]:
def normalized_sha(contents):
    return hashlib.sha256(contents.replace(b"\r\n", b"\n")).hexdigest()

for name, (expected_sha, source_text) in bundled_sources.items():
    source_bytes = source_text.encode("utf-8")
    if normalized_sha(source_bytes) != expected_sha:
        raise ValueError(f"Embedded source checksum mismatch for {name}")
    target = SOURCE / name
    try:
        with target.open("xb") as stream:
            stream.write(source_bytes)
        origin = "restored from notebook"
    except FileExistsError:
        origin = "existing local file"
    actual_sha = normalized_sha(target.read_bytes())
    if origin == "restored from notebook" and actual_sha != expected_sha:
        raise OSError(f"Written source checksum mismatch for {name}")
    print(f"{name}: {origin}, SHA-256 {actual_sha}")
    if actual_sha != expected_sha:
        print(f"  Using the local revision. Notebook revision is {expected_sha}.")

## Choose the run
The short default tests the pipeline. Set `REFERENCE_RECIPE = True` for the 6,000-step CPU reference configuration. Repeat with seeds 1337, 1338 and 1339. Keep the configuration fixed for device comparisons. A TPU comparison runs each mode in a fresh child process. Give the TPU a descriptive hardware label from your runtime and use the same runtime for all three measurements.

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

RUN_TPU = False
TPU_LABEL = ""  # Example: "v6e-1" if that is the hardware you allocated.
REFERENCE_RECIPE = False
SEED = 1337  # Repeat the reference with 1338 and 1339.
STEPS = 6000 if REFERENCE_RECIPE else 256
BATCH_SIZE = 128
WIDTH = 128 if REFERENCE_RECIPE else 64
SLICES = 64 if REFERENCE_RECIPE else 32
BLOCK_SIZE = 50 if REFERENCE_RECIPE else 32
WEIGHT = 0.001

session = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
WORK = SOURCE / "notebook-runs" / session
RUNS = WORK / "runs"
OUT = WORK / "report"
RUNS.mkdir(parents=True)
print(WORK)

## Prepare the Python environment
In Colab, start with a CPU runtime for the first pass. For TPU measurements, switch the runtime hardware, enable the TPU flag, and run all cells again. The installation cell follows the [Cloud TPU PyTorch setup](https://docs.cloud.google.com/tpu/docs/pytorch-on-tpu). Python, PyTorch and backend versions are recorded by the benchmark. TPU availability and supported software wheel combinations depend on the chosen runtime.

In [ ]:
packages = [name for name in ("numpy", "matplotlib") if importlib.util.find_spec(name) is None]
if RUN_TPU:
    if not TPU_LABEL.strip():
        raise ValueError("Set TPU_LABEL to the hardware reported by your TPU runtime.")
    if importlib.util.find_spec("torch_tpu") is None:
        packages.append("torch_tpu")
elif importlib.util.find_spec("torch") is None:
    packages.append("torch")
if packages:
    subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)
print("Run directory:", RUNS)

## Train the control and the regularized model
The first two subprocesses run the CPU eager controls. On a TPU runtime the same cell also runs TPU strict eager, fused eager, compiled execution and the eager control with SIGReg disabled. A TPU backend error is preserved in its run directory for inspection. The CPU control must return successfully for the experiment to proceed.

In [ ]:
import json

def run_one(device, mode, weight):
    label = f"{device}-{mode}-{'alignment' if weight == 0 else 'sigreg'}"
    destination = RUNS / label
    command = [
        sys.executable, str(SOURCE / "run_experiment.py"), "--device", device, "--mode", mode,
        "--output", str(destination), "--seed", str(SEED), "--steps", str(STEPS),
        "--width", str(WIDTH), "--batch-size", str(BATCH_SIZE), "--slices", str(SLICES),
        "--block-size", str(BLOCK_SIZE), "--weight", str(weight),
        "--comparison-group", session,
    ]
    if device == "tpu":
        command += ["--device-label", TPU_LABEL]
    completed = subprocess.run(command, cwd=SOURCE)
    record_path = destination / "result.json"
    if not record_path.exists():
        raise RuntimeError(f"{label} produced no result.json. Review subprocess output above.")
    record = json.loads(record_path.read_text())
    print(label, record["status"], record.get("error") or "")
    if device == "cpu":
        completed.check_returncode()

run_one("cpu", "eager", 0.0)
run_one("cpu", "eager", WEIGHT)
if RUN_TPU:
    for mode in ("eager", "fused", "compile"):
        run_one("tpu", mode, WEIGHT)
    run_one("tpu", "eager", 0.0)

## Inspect the results
**First step.** The timer waits for the device before and after the first training step in a fresh process. This wall time includes host dispatch, device execution and any first-use compilation. Persistent caches can affect it.

**Warm throughput.** Each later block is synchronized at its boundaries. The steady steps/s estimate starts after the first such block. It includes Python dispatch, the forward and backward passes, and AdamW. Input staging happens outside the timer. A host-side dispatch bottleneck is one possible explanation for a difference between modes. Isolating that cause would require a profiler trace.

**Break-even.** Compare cumulative times at matching block boundaries. If a mode catches up between two boundaries, those steps bracket the observed crossover. A short run can end before either mode catches up.

**Learning check.** Read held-out R² alongside representation variance. A low alignment loss can accompany collapsed embeddings. Compare the regularized model with alignment alone and with the raw observation baseline.

In [ ]:
from IPython.display import Markdown, Image, display

subprocess.run([sys.executable, str(SOURCE / "report_results.py"),
                "--runs", str(RUNS), "--out", str(OUT)], check=True)
# Display inline figures even if the notebook server has a different document root.
import re
for part in re.split(r"(!\[[^\]]*\]\([^)]*\.png\))", (OUT / "report.md").read_text(encoding="utf-8")):
    match = re.fullmatch(r"!\[[^\]]*\]\(([^)]*\.png)\)", part)
    if match:
        display(Image(filename=str(OUT / match.group(1))))
    elif part.strip():
        display(Markdown(part))

## Keep the evidence
The archive includes all raw JSON records, scatter plots and the report. Download it before releasing or changing the runtime. Repeat with independent seeds to measure variability. Preserve failures and include the training budget, seed, cache conditions and exact versions alongside any performance figure.

In [ ]:
import shutil

archive = shutil.make_archive(str(WORK) + "-results", "zip", root_dir=WORK)
print(archive)
# In Colab, download the archive by uncommenting the next two lines:
# from google.colab import files
# files.download(archive)